In [ ]:
import sys, os
sys.path.append("..")
from MRE import Job
from collections import defaultdict

# ============================
# PARÁMETROS
# ============================
ALFA = 0.1
PH_INICIAL = 1.0
ERROR = 0.1
MAX_ITER = 20

inputDir = "../Datasets/TP1/input/"
outputDir = "../Datasets/TP1/output/"

# ============================
# 1) MAP–REDUCE: Filtrar combates (menor tiempo)
# ============================
def fmap_filtrar(_, value, context):
    parts = value.strip().split()
    if len(parts) < 4:
        return
    retador, retado, puntos, tiempo = parts
    context.write(f"{retador}-{retado}", (float(puntos), float(tiempo)))

def fred_filtrar(key, values, context):
    mejor = None
    for v in values:
        if mejor is None or v[1] < mejor[1]:
            mejor = v
    context.write(key, mejor)


# ============================
# 2) MAP–REDUCE: Puntaje promedio (PP)
# ============================
def fmap_pp(key, value, context):
    retador, retado = key.split('-')
    puntos, tiempo = value
    context.write(retador, ("S", puntos))
    context.write(retado, ("R", 0.0))  # asegura existencia del retado

def fred_pp(jugador, values, context):
    suma, cuenta = 0.0, 0
    for tag, puntos in values:
        if tag == "S":
            suma += puntos
            cuenta += 1
    if cuenta == 0:
        cuenta = 1
        suma = 1.0
    context.write(jugador, suma / cuenta)


# ============================
# 3) MAP–REDUCE iterativo: Puntaje heroico (PH)
# ============================
def fmap_ph(key, value, context):
    retador, retado = key.split('-')
    puntos, tiempo = value
    pp = context["pp"]
    ph_prev = context["ph_prev"]

    pp_i = pp.get(retador, 1.0)
    pp_j = pp.get(retado, 1.0)
    ph_j = ph_prev.get(retado, PH_INICIAL)

    contrib = ph_j * (pp_i / pp_j)
    context.write(retador, contrib)

def fred_ph(jugador, values, context):
    suma = sum(values)
    nuevo_ph = ALFA * suma + (1 - ALFA)
    context.write(jugador, nuevo_ph)


# ============================
# FUNCIONES AUXILIARES
# ============================
def leer_resultados(path):
    """
    Lee los resultados del Job (output.txt) y devuelve un dict {clave: valor}
    """
    output_file = os.path.join(path, "output.txt")
    data = {}
    if not os.path.exists(output_file):
        return data
    with open(output_file, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                key = parts[0]
                try:
                    val = float(parts[1])
                except ValueError:
                    val = parts[1]
                data[key] = val
    return data


# ============================
# EJECUCIÓN PRINCIPAL
# ============================
if __name__ == "__main__":
    # Paso 1: Filtrar duplicados
    job1_out = os.path.join(tmpDir, "filtrado")
    job1 = Job(inputDir, job1_out, fmap_filtrar, fred_filtrar)
    job1.waitForCompletion()

    # Paso 2: Calcular PP
    job2_out = os.path.join(tmpDir, "pp")
    job2 = Job(job1_out, job2_out, fmap_pp, fred_pp)
    job2.waitForCompletion()
    pp = leer_resultados(job2_out)

    # Paso 3: Iterativo – calcular PH
    ph = defaultdict(lambda: PH_INICIAL)
    for it in range(MAX_ITER):
        job3_out = os.path.join(tmpDir, f"ph_iter{it+1}")
        params = {"pp": pp, "ph_prev": ph}
        job3 = Job(job1_out, job3_out, fmap_ph, fred_ph)
        job3.setParams(params)
        job3.waitForCompletion()
        nuevo_ph = leer_resultados(job3_out)

        # Convergencia
        if not nuevo_ph:
            break
        max_dif = max(abs(nuevo_ph[j] - ph.get(j, 0.0)) for j in nuevo_ph)
        print(f"[Iter {it+1}] max_dif={max_dif:.4f}")
        ph = nuevo_ph
        if max_dif < ERROR:
            print(f"[INFO] Convergencia alcanzada en iteración {it+1}")
            break

    # Paso 4: Top 10
    top10 = sorted(ph.items(), key=lambda x: x[1], reverse=True)[:10]
    print("\nTop 10 jugadores por puntaje heroico:")
    for j, p in top10:
        print(f"Jugador={j}, PH={p:.2f}")
